In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

### 1.3.1

In [4]:
# 1. Загрузка данных (используем ваш формат)
df = pd.read_csv("./tipa.csv")

# 2. Выборочные средние
mean_X = df['X'].mean()
mean_Y = df['Y'].mean()

# 3. Обучение модели линейной регрессии
# X должен быть двумерным массивом для sklearn
X_matrix = df[['X']] 
y = df['Y']

model = LinearRegression()
model.fit(X_matrix, y)

# 4. Коэффициенты
coef_slope = model.coef_[0]      # Коэффициент наклона (при X)
coef_intercept = model.intercept_ # Свободный член (сдвиг)

# 5. Оценка точности (R^2)
y_pred = model.predict(X_matrix)
r2 = r2_score(y, y_pred)

# Вывод результатов (с округлением до 3-4 знаков, как обычно требуют)
print(f"Среднее X: {mean_X:.4f}")
print(f"Среднее Y: {mean_Y:.4f}")
print(f"Коэффициент наклона (w1): {coef_slope:.4f}")
print(f"Свободный член (w0): {coef_intercept:.4f}")
print(f"Точность (R^2): {r2:.4f}")

Среднее X: 11.8000
Среднее Y: 25.1000
Коэффициент наклона (w1): 1.6697
Свободный член (w0): 5.3980
Точность (R^2): 0.8525


### 1.3.2

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 1. Загрузка данных
df = pd.read_csv('candy-data.csv')

# 2. Выбор предикторов и отклика
features = [
    'chocolate', 'fruity', 'caramel', 'peanutyalmondy', 'nougat',
    'crispedricewafer', 'hard', 'bar', 'pluribus', 'sugarpercent', 'pricepercent'
]
target = 'winpercent'

# 3. Исключаем две конфеты из тренировочного набора
exclude = ['Boston Baked Beans', 'Peanut M&Ms']
train_df = df[~df['competitorname'].isin(exclude)]

X_train = train_df[features]
y_train = train_df[target]

# 4. Обучение модели
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Предсказание для Boston Baked Beans
boston = df[df['competitorname'] == 'Boston Baked Beans']
pred_boston = model.predict(boston[features])[0]
true_boston = boston[target].values[0]
print(f"Boston Baked Beans: предсказание = {pred_boston:.3f}, истинное = {true_boston:.3f}")

# 6. Предсказание для Peanut M&Ms
peanut = df[df['competitorname'] == 'Peanut M&Ms']
pred_peanut = model.predict(peanut[features])[0]
true_peanut = peanut[target].values[0]
print(f"Peanut M&Ms: предсказание = {pred_peanut:.3f}, истинное = {true_peanut:.3f}")

Boston Baked Beans: предсказание = 44.005, истинное = 23.418
Peanut M&Ms: предсказание = 68.488, истинное = 69.484


### 1.3.3

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Загрузка
df = pd.read_csv('fish_train.csv')

# 2. Синтез признаков
df['Volume'] = df['Length1'] * df['Length2'] * df['Length3']
df['AspectRatio'] = df['Length3'] / df['Length1']
df['HeightWidthRatio'] = df['Height'] / df['Width']
df['Area'] = df['Length2'] * df['Height']

# 3. Признаки и цель
numeric_features = [
    'Length1', 'Length2', 'Length3', 'Height', 'Width',
    'Volume', 'AspectRatio', 'HeightWidthRatio', 'Area'
]
categorical_features = ['Species']
target = 'Weight'

X = df[numeric_features + categorical_features]
y = df[target]

# 4. Препроцессинг: масштабирование + one-hot для Species
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

# 5. Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Модели
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }

# 7. Вывод
print(pd.DataFrame(results).T.round(4))

                      MAE     RMSE      R2
LinearRegression  30.5717  47.8405  0.9728
RandomForest      27.5626  45.7361  0.9751
GradientBoosting  35.7848  53.4437  0.9660
